# geom_pie()

A pie chart is a circular statistical graphic, which is divided into slices to illustrate numerical proportion.

1. [Default presentation](#1.-Default-presentation)

  1.1. [Basic pie chart](#1.1.-Basic-pie-chart)

  1.2. [Improve appearance](#1.2.-Improve-appearance)   

  1.3. [Adding labels to pie sectors](#1.3.-Adding-labels-to-pie-sectors) 

  1.4. [Use "count2d" statistical transformation](#1.4.-Use-"count2d"-statistical-transformation)


2. [Pie size depending on data](#2.-Pie-size-depending-on-data)


3. [Explode](#3.-Explode)

In [1]:
from lets_plot import *
from lets_plot.mapping import *

In [2]:
LetsPlot.setup_html() 

In [3]:
w,h = 400,250

df = {
  'name':  ['a', 'b', 'c', 'd', 'b'],
  'value': [40,   90, 10,  50,  20 ],
}

p = ggplot(df) + ggsize(w,h)

## 1. Default presentation

### 1.1. Basic pie chart

Use  "identity" statistical transformation to leave the data unchanged.

In [4]:
p + geom_pie(aes(slice='value', fill='name'), stat='identity')

### 1.2. Improve appearance

- add stroke (`stroke` and `stroke_color`)
- make the pie bigger (`size`)
- add hole to draw donut-like chart (`hole`)
- use blank theme (remove axis and grid)
- use better colors

In [5]:
p + \
    geom_pie(aes(slice='value', fill='name'), stat='identity',
            size = 20, stroke = 1, hole = 0.5) + \
    theme_void()

### 1.3. Adding labels to pie sectors

Let's label the sectors with their names - configure annotations via `layer_labels()` function: 

In [6]:
p + \
    geom_pie(aes(slice='value', fill='name'), stat='identity',
            size = 20, stroke = 1, hole = 0.5,
            labels = layer_labels().line('@name').size(16)) + \
    theme_void() + \
    theme(legend_position='none')

### 1.4. Use "count2d" statistical transformation

`geom_pie()` uses `count2d` stat by default. 
It allows to make a slice sizes proportional to the number of cases in each group  (or if the weight aesthetic is supplied, the sum of the weights). Also `count2d` provides variables for proportion ('..prop..') and proportion in percent ('..proppct..'). 

Using `layer_tooltips()` prepare the information for tooltips by adding the variables provided by 'count2d':

In [7]:
tooltip_content = (
    layer_tooltips()
        .line('count|@{..count..} (@{..prop..})')
        .line('total|@{..sum..}')
        .format('..prop..', '.0%')
        .format('..count..', '.1f')
        .format('..sum..', '.1f')
)

Apply 'count2d' to get slices proportional to the number of cases.

In [8]:
p + \
    geom_pie(aes(fill='name'),
             size = 20, stroke = 1, hole = 0.5,
             labels = layer_labels().line('@name').size(16), 
             tooltips=tooltip_content) + \
    theme_void() + \
    theme(legend_position='none')

Compute weighted sum instead of simple count with aesthetic `weight`.

In [9]:
p + \
    geom_pie(aes(fill='name', weight='value'),
             size = 20, stroke = 1, hole = 0.5,
             labels = layer_labels().line('@name').size(16), 
             tooltips=tooltip_content) + \
    theme_void() + \
    theme(legend_position='none')

Order sectors by count.

The following ordering rule is used for the pie chart: the first slice goes to the left of 12 o'clock and others go clockwise.

In [10]:
p + \
    geom_pie(aes(fill=as_discrete('name', order_by='..count..'), weight='value'),
             size = 20, stroke = 1, hole = 0.5,
             labels = layer_labels().line('@name').size(16), 
             tooltips=tooltip_content) + \
    theme_void() + \
    theme(legend_position='none')

## 2. Pie size depending on data

Make the size of the pie chart dependent on the data: map total count ('..sum..' variable) to the `size`.

Note that it has its own special representation of the size in the legend.

In [11]:
df2 = {
    'x': [1, 1, 1, 1, 1, 1.5, 1.5,   2,   2,   2 ],
    'y': [1, 1, 1, 1, 1,   2,   2, 1.5, 1.5, 1.5],
    's': [3, 1, 2, 1, 4,   1,   3,   3,   3,  1],
    'n': ['a', 'b', 'a', 'c', 'a',  'a', 'b', 'c', 'a',  'b']
}

ggplot(df2) + \
    geom_pie(aes('x', 'y', fill='n', weight='s', size='..sum..'), hole=0.3, 
             tooltips=tooltip_content) + \
    xlim(0.5,2.5) + ylim(0.5,2.5) 

Mapping `fill` and `size` to the same variable:

In [12]:
ggplot({'n': ["a", "b", "c"], 's': [1, 2, 3]}) + \
    geom_pie(aes(fill='n', slice='s', size='n'), stat="identity") + \
    theme_void()

## 3. Explode

Use values to explode slices away from their center point, detaching it from the main pie.

In [13]:
length = {
  'name' : ['20-50 km', '50-75 km', '10-20 km', '75-100 km', '3-5 km', '7-10 km', '5-7 km', '>100 km', '2-3 km'],
  'count': [1109, 696, 353, 192, 168, 86, 74, 65, 53],
  'explode': [0, 0, 0, 0.1, 0.1, 0.2, 0.3, 0.4, 0.6]
}

ggplot(length) + \
    geom_pie(aes(fill='name', slice='count', explode='explode'), stat='identity',
            stroke=1, color='black', size=20) + \
    theme_void() + \
    scale_fill_gradient(low='dark_blue', high='light_green')

In [14]:
calories = {
    'slice': [35, 25, 25, 15],
    'label': ["Apples", "Bananas", "Cherries", "Dates"],
    'explode': [0.1, 0, 0, 0]
}

p4 = ggplot(calories) + \
    theme_void() + \
    ggsize(w, h)

p_pie = p4 + \
    geom_pie(aes(fill='label', slice='slice', explode='explode'), stat='identity', size=18)

p_donut = p4 + \
    geom_pie(aes(fill='label', slice='slice', explode='explode'), stat='identity', hole=0.8, size=18)

bunch = GGBunch()
bunch.add_plot(p_pie + theme(legend_position='none'), 0, 0)
bunch.add_plot(p_donut, w, 0)
bunch.show()